# Course 1 - Modul 3: Exploratory Data Analysis (EDA) & Feature Engineering
**Mata Kuliah**: Data Mining  
**Identifikasi Progress**: Minggu 3 (`c1-m3`)  
**Format File**: Jupyter Lab Notebook (`.ipynb`)  
**Lingkup Materi**: Eksplorasi Data Awal (EDA), Statistik Ringkasan & Sampling, Visualisasi Lanjutan (Pandas Groupby Plot, Seaborn Pairplot, Hexbin Jointplot, FacetGrid), Transformasi Variabel (Log, Box-Cox, Polinomial), Encoding Fitur Kategorikal (Nominal & Ordinal), serta Penskalaan Fitur (*StandardScaler*, *MinMaxScaler*, *RobustScaler*).

---

## 🎯 Capaian Pembelajaran Minggu 3:
1. Memahami esensi **Exploratory Data Analysis (EDA)** sebagai *"percakapan awal dengan data"* untuk mengidentifikasi karakteristik, pola, anomali, dan korelasi antar variabel.
2. Menguasai teknik kalkulasi statistik ringkasan dan strategi pengambilan sampel (*Simple Random Sampling* dan *Stratified Sampling* untuk data tidak seimbang).
3. Menerapkan ekosistem visualisasi data Python: Matplotlib, Pandas Plotting wrapper, dan Seaborn visualisasi tingkat tinggi (*Pairplot*, *Hexbin Jointplot*, dan *FacetGrid*).
4. Memahami asumsi model linear mengenai normalitas residual dan menerapkan teknik transformasi variabel (*Log Transform*, `log1p`, *Box-Cox Transformation*, dan *Polynomial Features*).
5. Mampu melakukan rekayasa fitur (*Feature Engineering*) melalui encoding variabel kategorikal (*One-Hot Encoding* / `get_dummies`, *Binary Encoding*, dan *Ordinal Encoding*).
6. Menguasai teknik penskalaan fitur numerik (*Feature Scaling*) menggunakan *StandardScaler*, *MinMaxScaler*, dan *RobustScaler*, serta memahami implikasinya pada algoritma machine learning berbasis jarak.


---
## 1. Setup Lingkungan & Import Library


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Scikit-learn preprocessing
from sklearn.preprocessing import (
    StandardScaler, 
    MinMaxScaler, 
    RobustScaler, 
    OneHotEncoder, 
    OrdinalEncoder, 
    LabelBinarizer,
    PolynomialFeatures
)
from sklearn.model_selection import train_test_split

# Konfigurasi plotting
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', None)

print("Semua pustaka berhasil diimpor.")


---
## 2. Eksplorasi Data Awal (EDA) & Statistik Ringkasan

EDA adalah pendekatan analitis untuk meringkas karakteristik utama dataset sebelum membangun model prediktif.

### 2.1. Memuat Dataset & Ringkasan Statistik
Kita menggunakan berkas `data/iris_data.csv` yang berisi fitur morfologi bunga iris.


In [ ]:
# Memuat data
df_iris = pd.read_csv('data/iris_data.csv')
print("Bentuk Dataset Iris:", df_iris.shape)
print("\nLima Baris Pertama:")
display(df_iris.head())

print("\n=== Ringkasan Statistik Deskriptif ===")
display(df_iris.describe().round(2))

print("\n=== Matriks Korelasi Fitur Numerik ===")
display(df_iris.drop(columns='species').corr().round(2))


---
### 2.2. Teknik Pengambilan Sampel (*Sampling from DataFrames*)

Ada kalanya dataset berukuran sangat besar sehingga komputasi menjadi lambat, atau kita memerlukan sampel yang merepresentasikan proporsi kelas tertentu secara presisi:
1. **Simple Random Sampling**:
   - `df.sample(n=5, replace=False)`: Memilih 5 baris acak tanpa pengembalian (setiap baris hanya muncul satu kali).
2. **Stratified Sampling**:
   - Mengambil sampel dengan tetap mempertahankan proporsi kelas target (sangat penting untuk dataset *imbalanced*, seperti deteksi penipuan atau penyakit langka).


In [ ]:
# 1. Simple Random Sampling (5 baris tanpa pengembalian)
sample_random = df_iris.sample(n=5, replace=False, random_state=42)

print("Sampel Acak (Semua baris, 3 kolom terakhir menggunakan .iloc[:, -3:]):")
display(sample_random.iloc[:, -3:])


In [ ]:
# 2. Stratified Sampling berdasarkan kolom species
# Mengambil 20% sampel dari setiap kelas spesies secara proporsional
sample_stratified = df_iris.groupby("species").sample(frac=0.2, random_state=42)

print("Proporsi Sampel per Spesies:")
print(sample_stratified["species"].value_counts())
print(f"Total Baris Sampel Terstratifikasi: {len(sample_stratified)}")


---
## 3. Visualisasi Data untuk EDA

Python memiliki 3 tingkatan pustaka visualisasi:
1. **Matplotlib**: Pustaka dasar dengan fleksibilitas penuh untuk kustomisasi tingkat rendah.
2. **Pandas Plotting**: Pembungkus (*wrapper*) mudah di atas Matplotlib untuk eksplorasi cepat.
3. **Seaborn**: Pustaka tingkat tinggi yang dirancang khusus untuk visualisasi statistik yang estetis.

### 3.1. Pandas GroupBy Plotting
Menghitung rata-rata setiap fitur morfologi berdasarkan spesies, lalu memvisualisasikannya dengan kustomisasi warna dan ukuran kanvas.


In [ ]:
# Menghitung rata-rata fitur berdasarkan spesies
mean_by_species = df_iris.groupby('species').mean()
print("Rata-rata Fitur per Spesies:")
display(mean_by_species)

# Visualisasi Bar Plot menggunakan sintaks Pandas
ax = mean_by_species.plot(
    kind='bar',
    figsize=(8, 5),
    fontsize=10,
    color=['#e74c3c', '#3498db', '#2ecc71', '#9b59b6'], # Merah, Biru, Hijau, Ungu
    edgecolor='black'
)

plt.title('Rata-rata Karakteristik Morfologi per Spesies Iris', fontsize=12, fontweight='bold', pad=10)
plt.xlabel('Spesies', fontsize=10)
plt.ylabel('Nilai Rata-rata (cm)', fontsize=10)
plt.xticks(rotation=0)
plt.legend(title='Fitur', fontsize=9)
plt.tight_layout()
plt.show()


---
### 3.2. Seaborn Pairplot
`sns.pairplot()` menghasilkan matriks scatter plot antara seluruh pasangan variabel numerik, dilengkapi kurva densitas pada diagonal utama dengan diferensiasi warna berdasarkan target (`hue='species'`).


In [ ]:
# Visualisasi Seaborn Pairplot
pair_fig = sns.pairplot(
    df_iris, 
    hue='species', 
    palette='Set2', 
    diag_kind='kde',
    height=2.2
)
pair_fig.fig.suptitle('Seaborn Pairplot: Distribusi Bivariat dan Pemisahan Spesies Iris', y=1.02, fontsize=12, fontweight='bold')
plt.show()


---
### 3.3. Seaborn Hexbin Joint Plot
Ketika menganalisis ribuan titik data kontinu, scatter plot sering mengalami masalah *overplotting* (titik bertumpuk). `sns.jointplot(kind='hex')` mengelompokkan data ke dalam kisi heksagonal 2D di mana warna heksagon yang lebih gelap merepresentasikan densitas observasi yang lebih padat, disertai histogram marginal di sisi atas dan kanan.


In [ ]:
# Hexbin Joint Plot antara Sepal Length vs Sepal Width
joint_hex = sns.jointplot(
    data=df_iris,
    x='sepal_length',
    y='sepal_width',
    kind='hex',
    color='#2980b9',
    gridsize=18,
    marginal_kws=dict(bins=15, fill=True)
)
joint_hex.fig.suptitle('Hexbin Plot: Kerapatan Densitas 2D Sepal Length vs Sepal Width', y=1.02, fontsize=11, fontweight='bold')
plt.show()


---
### 3.4. Seaborn FacetGrid
`sns.FacetGrid` memungkinkan pemisahan dataset menjadi sub-grafik multi-panel berdasarkan nilai kategori tertentu.


In [ ]:
# FacetGrid: Memetakan Histogram Sepal Width untuk Setiap Spesies
grid_width = sns.FacetGrid(df_iris, col='species', margin_titles=True, height=3.5, aspect=1.1)
grid_width.map(plt.hist, 'sepal_width', bins=10, color='darkcyan', edgecolor='black')
grid_width.fig.subplots_adjust(top=0.8)
grid_width.fig.suptitle('Distribusi Sepal Width per Spesies (FacetGrid)', fontsize=12, fontweight='bold')
plt.show()

# FacetGrid: Memetakan Histogram Sepal Length untuk Setiap Spesies
grid_length = sns.FacetGrid(df_iris, col='species', margin_titles=True, height=3.5, aspect=1.1)
grid_length.map(plt.hist, 'sepal_length', bins=10, color='darkorange', edgecolor='black')
grid_length.fig.subplots_adjust(top=0.8)
grid_length.fig.suptitle('Distribusi Sepal Length per Spesies (FacetGrid)', fontsize=12, fontweight='bold')
plt.show()


---
## 4. Transformasi Variabel & Normalitas Residual

Model statistik parametrik (seperti Regresi Linear) mengasumsikan bahwa **residual model berdistribusi normal**. Namun dalam dunia nyata, data sering kali memiliki kemiringan (*skewness*) yang tinggi:
- **Positif / Right-Skewed**: Ekor data memanjang ke kanan (sebagian besar data bernilai kecil, sedikit bernilai sangat besar, misal: pendapatan).
- **Negatif / Left-Skewed**: Ekor data memanjang ke kiri.

### Teknik Transformasi:
1. **Log Transformation**: Mengurangi *positive skew*. Rumus: $\log(x)$ atau `np.log1p(x)` $= \log(1 + x)$ untuk menangani data bernilai nol.
2. **Box-Cox Transformation**: Menemukan parameter eksponen $\lambda$ optimal secara otomatis untuk mentransformasikan data non-normal menjadi mendekati normal:
   $$y^{(\lambda)} = \begin{cases} \frac{y^\lambda - 1}{\lambda}, & \text{jika } \lambda \neq 0 \\ \ln(y), & \text{jika } \lambda = 0 \end{cases}$$
3. **Polynomial Features**: Menghasilkan interaksi non-linear ($x_1^2, x_1 x_2, x_2^2$).


In [ ]:
# Simulasi data dengan Right-Skewness Tinggi (Distribusi Eksponensial / Lognormal)
np.random.seed(42)
raw_skewed = np.random.exponential(scale=2.5, size=500)

# Menerapkan Transformasi Log dan Box-Cox
log_transformed = np.log1p(raw_skewed)
boxcox_transformed, optimal_lambda = stats.boxcox(raw_skewed + 1.0) # Ditambah konstanta jika ada nilai nol

print(f"Skewness Asli          : {stats.skew(raw_skewed):.2f}")
print(f"Skewness Log1p         : {stats.skew(log_transformed):.2f}")
print(f"Skewness Box-Cox       : {stats.skew(boxcox_transformed):.2f} (Optimal Lambda: {optimal_lambda:.2f})")


In [ ]:
# Visualisasi Perbandingan Distribusi Sebelum dan Sesudah Transformasi
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.histplot(raw_skewed, kde=True, color='crimson', ax=axes[0])
axes[0].set_title(f'Data Asli (Skew: {stats.skew(raw_skewed):.2f})', fontweight='bold')

sns.histplot(log_transformed, kde=True, color='forestgreen', ax=axes[1])
axes[1].set_title(f'Log1p Transformed (Skew: {stats.skew(log_transformed):.2f})', fontweight='bold')

sns.histplot(boxcox_transformed, kde=True, color='royalblue', ax=axes[2])
axes[2].set_title(f'Box-Cox Transformed (Skew: {stats.skew(boxcox_transformed):.2f})', fontweight='bold')

plt.tight_layout()
plt.show()


#### Transformasi Polinomial (*Polynomial Features*)


In [ ]:
# Demonstrasi Pembuatan Fitur Polinomial Derajat 2
sample_features = np.array([[2.0, 3.0], [4.0, 5.0], [6.0, 7.0]])
poly = PolynomialFeatures(degree=2, include_bias=False)
poly_result = poly.fit_transform(sample_features)

print("Fitur Asli [x1, x2]:\n", sample_features)
print("\nFitur Polinomial [x1, x2, x1^2, x1*x2, x2^2]:\n", poly_result)
print("Nama Fitur Baru:", poly.get_feature_names_out(['x1', 'x2']))


---
## 5. Encoding Fitur Kategorikal

Algoritma machine learning membutuhkan input numerik. Data kategorikal terbagi menjadi:
1. **Data Nominal (Unordered)**: Kategori tanpa tingkatan (misal: Warna, Status Pernikahan, Jenis Kelamin).
   - Menggunakan **One-Hot Encoding** (`pd.get_dummies` atau `OneHotEncoder(drop='first')`) atau **LabelBinarizer** untuk biner.
2. **Data Ordinal (Ordered)**: Kategori yang memiliki hierarki logis (misal: Pendidikan SD < SMP < SMA < S1, Rating Rendah < Sedang < Tinggi).
   - Menggunakan **OrdinalEncoder** dengan spesifikasi urutan kustom.


In [ ]:
# Membuat DataFrame simulasi data pelanggan
df_pelanggan = pd.DataFrame({
    'ID': [101, 102, 103, 104, 105],
    'Status_Menikah': ['Belum', 'Menikah', 'Menikah', 'Belum', 'Menikah'], # Biner
    'Metode_Bayar': ['Transfer', 'Kartu Kredit', 'E-Wallet', 'Transfer', 'Kartu Kredit'], # Nominal
    'Tingkat_Kepuasan': ['Rendah', 'Tinggi', 'Sedang', 'Sangat Tinggi', 'Sedang'] # Ordinal
})

print("DataFrame Asli:")
display(df_pelanggan)


In [ ]:
# 1. One-Hot Encoding pada Data Nominal (Metode_Bayar)
# Menggunakan pd.get_dummies dengan drop_first=True untuk menghindari Dummy Variable Trap
df_onehot = pd.get_dummies(df_pelanggan, columns=['Metode_Bayar'], drop_first=True, dtype=int)
print("DataFrame Setelah One-Hot Encoding (drop_first=True):")
display(df_onehot[['ID', 'Metode_Bayar_Kartu Kredit', 'Metode_Bayar_Transfer']])


In [ ]:
# 2. Ordinal Encoding pada Tingkat_Kepuasan dengan Hierarki Urutan
urutan_kepuasan = [['Rendah', 'Sedang', 'Tinggi', 'Sangat Tinggi']]
ord_encoder = OrdinalEncoder(categories=urutan_kepuasan)

df_pelanggan['Tingkat_Kepuasan_Encoded'] = ord_encoder.fit_transform(
    df_pelanggan[['Tingkat_Kepuasan']]
)

print("DataFrame Setelah Ordinal Encoding Terstruktur:")
display(df_pelanggan[['Tingkat_Kepuasan', 'Tingkat_Kepuasan_Encoded']])


---
## 6. Penskalaan Fitur (*Feature Scaling*)

Fitur numerik riil sering memiliki rentang nilai yang sangat berbeda jauh (misal: Usia $20-60$ vs Pendapatan $3.000.000 - 50.000.000$).
Algoritma berbasis jarak (K-Nearest Neighbors, K-Means Clustering, Support Vector Machines) dan optimasi gradien akan sangat terdistorsi oleh fitur berskala besar jika tidak diskalakan.

### Tiga Metode Penskalaan Utama:
1. **StandardScaler (Z-Score Standardization)**:
   $$z = \frac{x - \mu}{\sigma}$$
   Menghasilkan distribusi dengan $\text{mean} = 0$ dan $\text{std} = 1$. Sensitif terhadap outlier.
2. **MinMaxScaler (Normalisasi)**:
   $$x_{norm} = \frac{x - x_{min}}{x_{max} - x_{min}}$$
   Mengubah rentang nilai menjadi tepat $[0, 1]$. Sangat sensitif terhadap outlier.
3. **RobustScaler**:
   $$x_{robust} = \frac{x - \text{median}}{IQR}$$
   Menggunakan median dan Interquartile Range ($Q_{75} - Q_{25}$), sehingga **resisten terhadap outlier**.


In [ ]:
# Simulasi data dengan perbedaan skala drastis dan terdapat outlier
data_scaling = pd.DataFrame({
    'Harga_Produk': [15.0, 22.0, 18.0, 30.0, 25.0, 120.0], # Mengandung outlier 120
    'Jumlah_Stok': [12000, 15000, 11000, 18000, 14000, 13500]
})

print("Data Sebelum Penskalaan:")
display(data_scaling)

# Menerapkan 3 jenis scaler
scaler_std = StandardScaler()
scaler_minmax = MinMaxScaler()
scaler_robust = RobustScaler()

std_scaled = scaler_std.fit_transform(data_scaling)
minmax_scaled = scaler_minmax.fit_transform(data_scaling)
robust_scaled = scaler_robust.fit_transform(data_scaling)

print("\n=== Hasil StandardScaler (Mean=0, Std=1) ===")
display(pd.DataFrame(std_scaled, columns=data_scaling.columns).round(3))

print("\n=== Hasil MinMaxScaler (Rentang 0 hingga 1) ===")
display(pd.DataFrame(minmax_scaled, columns=data_scaling.columns).round(3))

print("\n=== Hasil RobustScaler (Tahan Outlier berbasis Median & IQR) ===")
display(pd.DataFrame(robust_scaled, columns=data_scaling.columns).round(3))


---
## 7. Kesimpulan & Rencana Progress Minggu 4

### 📝 Kesimpulan Minggu 3:
- Berhasil mempraktekkan alur **Exploratory Data Analysis (EDA)** terpadu: kalkulasi statistik ringkasan, korelasi, dan teknik sampling (acak vs terstratifikasi).
- Menguasai pembuatan visualisasi analitis lanjutan: **Pandas Groupby Bar Plot**, **Seaborn Pairplot**, **Hexbin 2D Joint Plot**, dan **FacetGrid Multi-panel**.
- Memahami asumsi normalitas model linear dan menerapkan transformasi **Log**, **Box-Cox**, dan **Polinomial**.
- Menguasai encoding data kategorikal (**One-Hot Encoding** untuk data nominal dan **Ordinal Encoding** berjenjang).
- Menguasai perbandingan **StandardScaler**, **MinMaxScaler**, dan **RobustScaler** serta memilih penskalaan yang tepat berdasarkan ada/tidaknya outlier.

### 🚀 Roadmap Minggu 4 (`c1-m4`):
Pada modul berikutnya, kita akan mendalami:
1. **Studi Kasus Telco Customer Churn (IBM Cognos)**: Analisis bivariat dan segmentasi churn.
2. **Fondasi Inferensial Statistik**: Pendekatan parametrik vs non-parametrik serta simulasi distribusi probabilitas (Uniform, Normal, Binomial, Poisson, Eksponensial).
3. **Frequentist vs Bayesian**: Simulasi koin bias, fungsi Likelihood, dan rasio likelihood.
4. **Pengujian Hipotesis (Neyman-Pearson)**: Error Tipe I & II, nilai p-value, Uji-t, Uji-F regresi, dan Koreksi Bonferroni.
